# Parcels environment : base

In [47]:
import parcels
import numpy as np
from datetime import timedelta
from glob import glob
import matplotlib.pyplot as plt
import xarray as xr
import os

def u2rho_2d (var_u):
    [Mp,L]=var_u.shape
    Lp=L+1
    Lm=L-1
    var_rho=np.zeros((Mp,Lp))
    var_rho[:,1:L-1]=0.5*(var_u[:,0:Lm-1]+var_u[:,1:L-1])
    var_rho[:,0]=var_rho[:,1]
    var_rho[:,Lp-1]=var_rho[:,-2]
    return var_rho
    
def v2rho_2d (var_v):
    [M,Lp]=var_v.shape
    Mp=M+1
    Mm=M-1
    var_rho=np.zeros((Mp,Lp))
    var_rho[1:M-1,:]=0.5*(var_v[0:Mm-1,:]+var_v[1:M-1,:])
    var_rho[0,:]=var_rho[1,:]
    var_rho[Mp-1,:]=var_rho[-2,:]
    return var_rho

def u2rho_3d (var_u):
    [N,Mp,L]=var_u.shape
    Lp=L+1
    Lm=L-1
    var_rho=np.zeros((N,Mp,Lp))
    var_rho[:,:,1:-1]=0.5*(var_u[:,:,1:]+var_u[:,:,:-1])
    var_rho[:,:,0]=var_rho[:,:,1]
    var_rho[:,:,-1]=var_rho[:,:,-2]
    return var_rho
    
def v2rho_3d (var_v):
    [N,M,Lp]=var_v.shape
    Mp=M+1
    Mm=M-1
    var_rho=np.zeros((N,Mp,Lp))
    var_rho[:,1:-1,:]=0.5*(var_v[:,1:,:]+var_v[:,:-1,:])
    var_rho[:,0,:]=var_rho[:,1,:]
    var_rho[:,-1,:]=var_rho[:,-2,:]
    return var_rho

def spheric_dist(lat1, lat2, lon1, lon2):
    """
    Compute the spherical distance between two points on Earth.
    
    Parameters:
    lat1, lat2 : array-like
        Latitude of the two points (in degrees).
    lon1, lon2 : array-like
        Longitude of the two points (in degrees).
    
    Returns:
    dist : array-like
        The spherical distance between the points (in meters).
    """
    
    # Earth radius in meters
    R = 6367442.76
    
    # Determine proper longitudinal shift
    l = np.abs(lon2 - lon1)
    l[l >= 180] = 360 - l[l >= 180]
    
    # Convert decimal degrees to radians
    deg2rad = np.pi / 180
    lat1 = lat1 * deg2rad
    lat2 = lat2 * deg2rad
    l = l * deg2rad
    
    # Compute the distances
    dist = R * np.arcsin(np.sqrt(((np.sin(l) * np.cos(lat2)) ** 2) + 
                                 ((np.sin(lat2) * np.cos(lat1)) - 
                                  (np.sin(lat1) * np.cos(lat2) * np.cos(l))) ** 2))
    
    return dist

def spheric_dist_one(lat1, lat2, lon1, lon2):
    """计算两个经纬度点之间的球面距离（标量版）"""
    # 处理经度差
    l = np.abs(lon2 - lon1)
    if l >= 180:
        l = 360 - l
    
    # 转换为弧度
    deg2rad = np.pi / 180
    lat1_rad = lat1 * deg2rad
    lat2_rad = lat2 * deg2rad
    l_rad = l * deg2rad
    
    # 球面距离公式
    distance = 6371 * np.arccos(
        np.sin(lat1_rad) * np.sin(lat2_rad) + 
        np.cos(lat1_rad) * np.cos(lat2_rad) * np.cos(l_rad)
    )
    return distance
def transunit_spher2flat(u,v,lat):
    v1=v*1852*60
    u1=u*1852*60*np.cos(lat*np.pi/180)
    return u1,v1

def trans_vel_roms(u,v,angle):
    # np.cos(angle*np.pi/180)
    # np.sin(angle*np.pi/180)
    u_east=u*np.cos(angle*np.pi/180)-v*np.sin(angle*np.pi/180)
    v_north=u*np.sin(angle*np.pi/180)+v*np.cos(angle*np.pi/180)
    return u_east,v_north

In [48]:
grid_dir='/meddy/simingzhang/Data/RB_iceland_data/'
wave_dir='/meddy/simingzhang/Data/Parcels_data/'
# nowave_dir='/meddy/simingzhang/Data/RB_iceland_data/iceland_no_wave/'
grdname='niskin2km_500m_grd.nc'
# hisname_w='z_niskin2km_his_hf_depth_500m_grd.0002.nc'
# hisname_nw='z_niskin2km_his_smooth_depth_500m_grd.0002.nc'

grdname=f'{grid_dir}{grdname}'
# wavename=f'{wave_dir}{hisname_w}'
# nowavename=f'{nowave_dir}{hisname_nw}'

# wavename=f'{wave_dir}wavecase_modified_cg.nc'
# nowavename=f'{wave_dir}nowavecase_modified_cg.nc'
wavename=f'{wave_dir}wavecase_modified_cg_tukey.nc'
nowavename=f'{wave_dir}nowavecase_modified_cg_tukey.nc'

In [49]:
grid=xr.open_dataset(grdname)
grid = grid.swap_dims({'eta_u': 'eta_rho','xi_v':'xi_rho'})

lon_rho=grid['lon_rho']
lat_rho=grid['lat_rho']
h=grid['h']
f=grid['f'].values
dx=1/grid['pm'].values
dy=1/grid['pn'].values
lonmin=np.min(lon_rho.values)
lonmax=np.max(lon_rho.values)
latmin=np.min(lat_rho.values)
latmax=np.max(lat_rho.values)
lonmin,lonmax,latmin,latmax
grid

<xarray.Dataset> Size: 19MB
Dimensions:    (one: 1, eta_rho: 287, xi_rho: 287, bath: 1, xi_u: 286,
                eta_v: 286, eta_psi: 286, xi_psi: 286)
Dimensions without coordinates: one, eta_rho, xi_rho, bath, xi_u, eta_v,
                                eta_psi, xi_psi
Data variables: (12/34)
    xl         (one) float64 8B ...
    el         (one) float64 8B ...
    depthmin   (one) float64 8B ...
    depthmax   (one) float64 8B ...
    spherical  (one) |S1 1B ...
    angle      (eta_rho, xi_rho) float64 659kB ...
    ...         ...
    lat_v      (eta_v, xi_rho) float64 657kB ...
    lat_psi    (eta_psi, xi_psi) float64 654kB ...
    mask_rho   (eta_rho, xi_rho) float64 659kB ...
    mask_u     (eta_rho, xi_u) float64 657kB ...
    mask_v     (eta_v, xi_rho) float64 657kB ...
    mask_psi   (eta_psi, xi_psi) float64 654kB ...
Attributes:
    title:    Solomon Model
    date:     08-Apr-2018
    type:     ROMS grid file

In [50]:
ds = xr.open_dataset(nowavename)
ds
# depth=ds['depth'].values

<xarray.Dataset> Size: 33GB
Dimensions:     (time: 2148, eta_rho: 287, xi_rho: 287, depth: 1, xi_u: 286,
                 eta_v: 286)
Coordinates:
  * time        (time) float64 17kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
    lon_rho     (eta_rho, xi_rho) float64 659kB ...
    lat_rho     (eta_rho, xi_rho) float64 659kB ...
  * depth       (depth) float32 4B -2.0
Dimensions without coordinates: eta_rho, xi_rho, xi_u, eta_v
Data variables: (12/45)
    Th1         (time, depth, eta_rho, xi_rho) float32 708MB ...
    Th2         (time, depth, eta_rho, xi_rho) float32 708MB ...
    Th3         (time, depth, eta_rho, xi_rho) float32 708MB ...
    Th4         (time, depth, eta_rho, xi_rho) float32 708MB ...
    Th5         (time, depth, eta_rho, xi_rho) float32 708MB ...
    Th6         (time, depth, eta_rho, xi_rho) float32 708MB ...
    ...          ...
    ocean_time  (time) float32 9kB ...
    u           (time, depth, eta_rho, xi_u) float32 705MB ...
    v           (time, depth, eta_v, xi_rho) float32 705MB ...
    w           (time, depth, eta_rho, xi_rho) float32 708MB ...
    u_rho       (time, depth, eta_rho, xi_rho) float64 1GB ...
    v_rho       (time, depth, eta_rho, xi_rho) float64 1GB ...
Attributes:
    CDI:          Climate Data Interface version 2.4.1 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Tue Sep 02 20:38:59 2025: cdo merge s2sflux_spec_all_smooth...
    CDO:          Climate Data Operators version 2.4.1 (https://mpimet.mpg.de...

In [51]:
u1=u2rho_3d(np.squeeze(ds['u'].values))
v1=v2rho_3d(np.squeeze(ds['v'].values))

In [ ]:
DX=np.tile(dx[np.newaxis,:,:],[2148,1,1])
DY=np.tile(dy[np.newaxis,:,:],[2148,1,1])

In [ ]:
vx=u2rho_3d((v1[:,:,1:]-v1[:,:,:-1])/(0.5*(DX[:,:,1:]+DX[:,:,:-1])))
uy=v2rho_3d((u1[:,1:,:]-u1[:,:-1,:])/(0.5*(DY[:,1:,:]+DY[:,:-1,:])))

ux=u2rho_3d((u1[:,:,1:]-u1[:,:,:-1])/(0.5*(DX[:,:,1:]+DX[:,:,:-1])))
vy=v2rho_3d((v1[:,1:,:]-v1[:,:-1,:])/(0.5*(DY[:,1:,:]+DY[:,:-1,:])))


In [ ]:
Ro=(vx-uy)/np.tile(f[np.newaxis,:,:],[2148,1,1])
divof=(ux+vy)/np.tile(f[np.newaxis,:,:],[2148,1,1])

In [ ]:
ds['Ro'] = xr.DataArray(
    Ro[:,np.newaxis,:,:],
    dims=ds['u_rho'].dims,  # 使用相同的维度顺序
    coords=ds['u_rho'].coords  # 使用相同的坐标
)
ds['divof'] = xr.DataArray(
    divof[:,np.newaxis,:,:],
    dims=ds['u_rho'].dims,  # 使用相同的维度顺序
    coords=ds['u_rho'].coords  # 使用相同的坐标
)

In [ ]:
ds

In [ ]:
all_vars = list(ds.data_vars)

# 移除除 Ro 和 div 外的所有变量
for var in all_vars:
    if var not in ['Ro', 'divof']:
        ds = ds.drop_vars(var)

In [ ]:
ds

In [ ]:
ds.to_netcdf(f'{wave_dir}nowavecase_Rodivof.nc')